# Solutions · 06 · Thixotropy

Worked solutions to the exercises of [notebook 06](../notebooks/06_thixotropy.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engrheo is missing): install it from GitHub
    import engrheo
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engrheo"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import optimize
from engrheo import datasets, thixotropy as th
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

In [2]:
cem = pd.read_csv(datasets.path("cement_paste_step_test.csv"), comment="#")
t, rate, tau = cem.time_s.to_numpy(), cem.shear_rate_1_s.to_numpy(), cem.shear_stress_Pa.to_numpy()
def model(p):
    prm = th.Params(*p)
    return th.simulate(prm, t, rate, lam0=th.equilibrium_structure(prm, rate[0]))["stress"]
use = rate > 0
p_fit = optimize.least_squares(lambda p: np.log(model(p)[use] / tau[use]), [10, 0.5, 1, 0.05, 0.05], bounds=(1e-6, np.inf),
                               x_scale="jac").x
prm = th.Params(*p_fit)

## Exercise 1

A cement grout line (50 mm, 200 m) stops for 10 min, and for 1 h. Using the fitted cement-paste model and $\Delta P = 4\tau_{restart}L/D$, estimate the pressure needed to restart flow in each case.

In [3]:
D, L = 0.05, 200.0
lam0 = th.equilibrium_structure(prm, 100.0)
for rest in (600, 3600):
    tau_r = prm.tau_y * (1 - (1 - lam0) * np.exp(-prm.k_build * rest))
    print(f"after {rest/60:3.0f} min: restart stress {tau_r:5.1f} Pa -> pressure {4*tau_r*L/D/1e5:5.2f} bar")

after  10 min: restart stress  24.9 Pa -> pressure  3.99 bar
after  60 min: restart stress  25.0 Pa -> pressure  4.00 bar


With this paste's build-up rate ($1/k_b$ ≈ 100 s) the structure is essentially rebuilt within ten minutes, so the
restart pressure is almost the same after 10 min and 1 h. Real cement also hydrates, so its yield stress keeps
rising for hours - the structural model covers only the reversible part.

## Exercise 2

For the paint, find the ramp time that gives the largest loop area. Relate it to the build-up and breakdown time scales $1/k_b$ and $1/(k_d\dot\gamma)$.

In [4]:
p_paint = th.Params(3.0, 0.08, 0.4, 0.02, 0.08)
ramps = np.logspace(-2, 3, 41)
areas = np.array([th.hysteresis_loop(p_paint, 200.0, tr)["area"] for tr in ramps])
best = ramps[np.argmax(areas)]
print(f"largest loop ({areas.max():.0f} Pa/s) for a ramp of {best:.2f} s")
print(f"breakdown time at the top rate 1/(k_break x 200) = {1/(0.08*200):.3f} s; build-up time 1/k_build = {1/0.02:.0f} s")

largest loop (3576 Pa/s) for a ramp of 0.18 s
breakdown time at the top rate 1/(k_break x 200) = 0.062 s; build-up time 1/k_build = 50 s


The loop is largest when the ramp lasts a few breakdown times: long enough for the structure to break down
during the ramp, too short for it to rebuild on the way down. The build-up time (50 s) is much longer, so for
ramps between seconds and minutes the down curve always lags - but the loop shrinks as the ramps get slower.

## Exercise 3

**Implement it yourself:** integrate the structure equation for the step test with the explicit Euler method (time step 1 s, then 10 s) and compare with the exact interval solution of `th.simulate`. When does Euler fail?

In [5]:
def euler(dt):
    ts = np.arange(0, t[-1] + dt / 2, dt)
    r = np.interp(ts, t, rate, left=rate[0])
    lam = np.empty_like(ts); lam[0] = th.equilibrium_structure(prm, 10.0)
    for i in range(ts.size - 1):
        lam[i + 1] = lam[i] + dt * (prm.k_build * (1 - lam[i]) - prm.k_break * r[i] * lam[i])
    return ts, lam
exact = th.simulate(prm, t, rate, lam0=th.equilibrium_structure(prm, 10.0))["structure"]
for dt in (0.5, 1.0, 10.0):
    ts, lam = euler(dt)
    err = np.max(np.abs(np.interp(t, ts, lam) - exact))
    print(f"Euler, dt = {dt:4.1f} s: max error {err:.3g}   (dt x rate constant at 100 1/s = {dt*(prm.k_build + prm.k_break*100):.2f})")

Euler, dt =  0.5 s: max error 0.0214   (dt x rate constant at 100 1/s = 1.00)
Euler, dt =  1.0 s: max error 0.157   (dt x rate constant at 100 1/s = 2.01)
Euler, dt = 10.0 s: max error 1.08e+25   (dt x rate constant at 100 1/s = 20.07)


Explicit Euler is stable only while the time step times the rate constant stays below 2. At 100 1/s the
rate constant is about 2 per second, so steps of 1 s sit at the edge and 10 s steps blow up. The structure
equation is **stiff** at high shear rates: the exact interval solution (or an implicit method) has no such
limit - one reason why the library integrates it exactly.